# 02 · Data Cleaning
Create a cleaned copy while leaving the raw CSV untouched. Run with the working directory set to `notebooks/`. Place the downloaded source at `../data/raw/BankChurners.csv`.

In [ ]:
from pathlib import Path
import pandas as pd
raw_path = Path('../data/raw/BankChurners.csv')
output_path = Path('../data/processed/cleaned_customers.csv')
if not raw_path.exists(): raise FileNotFoundError(f'{raw_path} missing; see ../data/README.md')
df_clean = pd.read_csv(raw_path)
print('Raw shape:', df_clean.shape)
df_clean.head()

## Remove irrelevant classifier output columns
The `Naive_Bayes_Classifier*` columns are precomputed classifier outputs rather than source customer attributes. Using them as predictors could leak the target or prior model decisions into future evaluation.

In [ ]:
classifier_cols = [c for c in df_clean.columns if c.startswith('Naive_Bayes_Classifier')]
print('Found:', classifier_cols)
df_clean = df_clean.drop(columns=classifier_cols)
print('Shape:', df_clean.shape)

## Duplicates and missing values
Remove exact duplicate rows, retaining the first occurrence. Keep `CLIENTNUM` for traceability, but exclude it from future predictive features. Report missingness without silent imputation; repeated identifiers require review.

In [ ]:
duplicate_rows = int(df_clean.duplicated().sum())
df_clean = df_clean.drop_duplicates().copy()
print('Exact duplicates removed:', duplicate_rows)
if 'CLIENTNUM' in df_clean: print('Duplicate CLIENTNUM remaining:', int(df_clean.CLIENTNUM.duplicated().sum()))
missing = df_clean.isna().sum().sort_values(ascending=False)
display(missing[missing.gt(0)].to_frame('missing_count'))
print('Total missing cells:', int(df_clean.isna().sum().sum()))

## Validate and save
Unusual values are reported, not silently clipped, recoded, or imputed. Review against dataset documentation before downstream use.

In [ ]:
expected = {'Attrition_Flag','CLIENTNUM','Customer_Age','Avg_Utilization_Ratio'}
print('Expected fields:', {c: c in df_clean.columns for c in sorted(expected)})
print('Final shape:', df_clean.shape)
display(df_clean.head())
display(df_clean.describe(include='all').T)
if 'Avg_Utilization_Ratio' in df_clean: print('Utilization outside [0,1]:', int((~df_clean.Avg_Utilization_Ratio.between(0,1)).fillna(False).sum()))
output_path.parent.mkdir(parents=True, exist_ok=True)
df_clean.to_csv(output_path, index=False)
print('Saved separate cleaned copy to', output_path)

## Cleaning decisions
- Drop columns starting with `Naive_Bayes_Classifier`.
- Drop exact duplicate rows only.
- Preserve `CLIENTNUM` for traceability; do not treat it as a model feature.
- Report but do not impute missing values or alter ranges/categories without a documented business decision.